# Embedding Model Fine-tuning with LoRA

Fine-tune **Qwen3-Embedding-0.6B** on Vietnamese medical text retrieval using LoRA.

## Qwen3-Embedding Configuration (Official Guidelines)
- **Padding Side**: LEFT (CRITICAL)
- **Pooling Strategy**: last_token (not mean)
- **Max Length**: 8192 tokens for training
- **Instruction Format**: `Instruct: {task}\nQuery: {query}`
- **Training Objective**: Contrastive learning (multiple negatives ranking)

Reference: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B

## 1. Setup

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoModel,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from sentence_transformers import losses
import wandb
import yaml

# Load LoRA config
with open("../configs/embedding_lora_config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Extract configuration (following Qwen3-Embedding guidelines)
model_config = config["model"]
lora_config = config["lora"]
data_config = config["data"]
training_config = config["training"]

MODEL_NAME = model_config["name"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CRITICAL: Qwen3-Embedding requirements
MAX_LENGTH = data_config["max_length"]  # 8192 from config
PADDING_SIDE = data_config["padding_side"]  # LEFT from config
POOLING_STRATEGY = data_config["pooling_strategy"]  # last_token from config
INSTRUCTION_TEMPLATE = data_config["instruction_template"]

print(f"Using device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"\nQwen3-Embedding Configuration:")
print(f"  - Max length: {MAX_LENGTH}")
print(f"  - Padding side: {PADDING_SIDE} (CRITICAL)")
print(f"  - Pooling: {POOLING_STRATEGY} (CRITICAL)")
print(f"  - Instruction template: {INSTRUCTION_TEMPLATE}")

## 2. Initialize W&B

In [ ]:
# Initialize wandb
wandb.init(
    project=training_config["wandb_project"],
    name=training_config["run_name"],
    config={
        **model_config,
        **lora_config,
        **data_config,
        **training_config
    }
)

print("W&B initialized successfully!")

## 3. Load Model and Tokenizer

In [ ]:
# Load tokenizer with LEFT padding (CRITICAL)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side=PADDING_SIDE  # "left" from config
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(DEVICE)

print(f"Model loaded: {MODEL_NAME}")
print(f"Tokenizer padding side: {tokenizer.padding_side}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 4. Load Dataset

In [ ]:
# Load dataset
dataset = load_dataset(data_config["dataset"])

# Split dataset if no test split exists (per Qwen guidelines)
if "test" not in dataset:
    print("No test split found. Creating 80/10/10 split...")
    train_val = dataset["train"].train_test_split(test_size=0.2, seed=42)
    val_test = train_val["test"].train_test_split(test_size=0.5, seed=42)
    
    train_dataset = train_val["train"]
    val_dataset = val_test["train"]
    test_dataset = val_test["test"]
else:
    train_dataset = dataset["train"]
    val_dataset = dataset.get("validation", dataset["train"].train_test_split(test_size=0.1, seed=42)["test"])
    test_dataset = dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\nSample:")
print(f"Question: {train_dataset[0]['question']}")
print(f"Answer: {train_dataset[0]['answer'][:100]}...")

## 5. Configure LoRA

In [ ]:
# LoRA configuration for embedding model
peft_config = LoraConfig(
    r=lora_config["r"],  # 32 from config (higher than generation model)
    lora_alpha=lora_config["alpha"],  # 64 from config
    target_modules=lora_config["target_modules"],
    lora_dropout=lora_config["dropout"],
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION  # For embedding models
)

# Apply LoRA to model
model = get_peft_model(model, peft_config)

# Print trainable parameters
model.print_trainable_parameters()

print("\nLoRA configuration applied successfully!")
print(f"LoRA rank: {lora_config['r']}")
print(f"LoRA alpha: {lora_config['alpha']}")

## 6. Prepare Contrastive Learning Data

In [ ]:
def format_with_instruction(text: str, task: str, is_query: bool = True) -> str:
    """Format text with Qwen3-Embedding instruction template.
    
    Template: Instruct: {task}\nQuery: {query}
    """
    if is_query:
        return INSTRUCTION_TEMPLATE.format(task=task, query=text)
    else:
        return text  # Documents don't need instruction


def create_contrastive_pairs(examples):
    """Create contrastive learning pairs (query, positive, negatives).
    
    For embedding fine-tuning, we use:
    - Query: Question with instruction
    - Positive: Corresponding answer
    - Negatives: Other answers in the batch (handled by loss function)
    """
    queries = []
    positives = []
    
    task = "Tìm kiếm thông tin y tế liên quan"  # Medical retrieval task
    
    for question, answer in zip(examples["question"], examples["answer"]):
        # Format query with instruction
        query = format_with_instruction(question, task, is_query=True)
        queries.append(query)
        
        # Positive document (no instruction)
        positives.append(answer)
    
    return {
        "query": queries,
        "positive": positives
    }


# Prepare contrastive pairs
print("Creating contrastive learning pairs...")
train_pairs = train_dataset.map(
    create_contrastive_pairs,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Creating training pairs"
)

val_pairs = val_dataset.map(
    create_contrastive_pairs,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Creating validation pairs"
)

print(f"Training pairs: {len(train_pairs)}")
print(f"Validation pairs: {len(val_pairs)}")
print(f"\nSample pair:")
print(f"Query: {train_pairs[0]['query'][:100]}...")
print(f"Positive: {train_pairs[0]['positive'][:100]}...")

## 7. Tokenize Data

In [ ]:
def tokenize_pairs(examples):
    """Tokenize query-positive pairs with Qwen3-Embedding configuration.
    
    CRITICAL:
    - Uses LEFT padding (tokenizer.padding_side="left")
    - Max length: 8192 tokens
    """
    # Tokenize queries
    query_inputs = tokenizer(
        examples["query"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )
    
    # Tokenize positives
    positive_inputs = tokenizer(
        examples["positive"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )
    
    return {
        "query_input_ids": query_inputs["input_ids"],
        "query_attention_mask": query_inputs["attention_mask"],
        "positive_input_ids": positive_inputs["input_ids"],
        "positive_attention_mask": positive_inputs["attention_mask"]
    }


# Tokenize datasets
print("Tokenizing datasets with LEFT padding and max_length=8192...")
tokenized_train = train_pairs.map(
    tokenize_pairs,
    batched=True,
    remove_columns=train_pairs.column_names,
    desc="Tokenizing training pairs"
)

tokenized_val = val_pairs.map(
    tokenize_pairs,
    batched=True,
    remove_columns=val_pairs.column_names,
    desc="Tokenizing validation pairs"
)

print(f"Tokenized training pairs: {len(tokenized_train)}")
print(f"Tokenized validation pairs: {len(tokenized_val)}")

## 8. Define Custom Contrastive Loss

In [ ]:
class ContrastiveTrainer(Trainer):
    """Custom trainer for contrastive learning with Qwen3-Embedding.
    
    Uses multiple negatives ranking loss (InfoNCE).
    """
    
    def compute_loss(self, model, inputs, return_outputs=False):
        """Compute contrastive loss.
        
        CRITICAL: Uses last_token pooling per Qwen3-Embedding guidelines.
        """
        # Extract query and positive inputs
        query_input_ids = inputs["query_input_ids"]
        query_attention_mask = inputs["query_attention_mask"]
        positive_input_ids = inputs["positive_input_ids"]
        positive_attention_mask = inputs["positive_attention_mask"]
        
        # Encode queries
        query_outputs = model(
            input_ids=query_input_ids,
            attention_mask=query_attention_mask
        )
        # Last token pooling (CRITICAL for Qwen3-Embedding)
        query_embeddings = query_outputs.last_hidden_state[:, -1, :]  # [batch_size, hidden_dim]
        
        # Encode positives
        positive_outputs = model(
            input_ids=positive_input_ids,
            attention_mask=positive_attention_mask
        )
        # Last token pooling
        positive_embeddings = positive_outputs.last_hidden_state[:, -1, :]  # [batch_size, hidden_dim]
        
        # Normalize embeddings
        query_embeddings = torch.nn.functional.normalize(query_embeddings, p=2, dim=1)
        positive_embeddings = torch.nn.functional.normalize(positive_embeddings, p=2, dim=1)
        
        # Compute similarity matrix (query vs all positives in batch)
        similarity_matrix = torch.matmul(query_embeddings, positive_embeddings.T)  # [batch_size, batch_size]
        
        # Scale by temperature
        temperature = 0.05
        similarity_matrix = similarity_matrix / temperature
        
        # Labels: diagonal elements are correct pairs
        labels = torch.arange(similarity_matrix.size(0)).to(similarity_matrix.device)
        
        # Cross-entropy loss (InfoNCE)
        loss = torch.nn.functional.cross_entropy(similarity_matrix, labels)
        
        return (loss, {"query_embeddings": query_embeddings, "positive_embeddings": positive_embeddings}) if return_outputs else loss


print("Custom contrastive trainer defined.")
print("Using multiple negatives ranking loss (InfoNCE) with last_token pooling.")

## 9. Training Arguments

In [ ]:
# Training arguments from config
training_args = TrainingArguments(
    output_dir=training_config["output_dir"],
    num_train_epochs=training_config["epochs"],
    per_device_train_batch_size=training_config["batch_size"],
    per_device_eval_batch_size=training_config["eval_batch_size"],
    gradient_accumulation_steps=training_config["gradient_accumulation_steps"],
    learning_rate=training_config["learning_rate"],
    warmup_ratio=training_config["warmup_ratio"],
    weight_decay=training_config["weight_decay"],
    logging_steps=training_config["logging_steps"],
    evaluation_strategy="steps",
    eval_steps=training_config["eval_steps"],
    save_strategy="steps",
    save_steps=training_config["save_steps"],
    save_total_limit=training_config["save_total_limit"],
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=training_config["fp16"],
    report_to="wandb",
    run_name=training_config["run_name"],
    push_to_hub=False
)

print("Training arguments configured.")
print(f"Output directory: {training_args.output_dir}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

## 10. Start Training

In [ ]:
# Initialize custom trainer
trainer = ContrastiveTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer
)

print("Starting training with contrastive learning...")
print("Using last_token pooling and multiple negatives ranking loss.")
print("\n" + "="*60)

# Train
trainer.train()

print("\n" + "="*60)
print("Training completed!")

## 11. Save Fine-tuned Model

In [ ]:
# Save LoRA adapter
output_path = training_config["output_dir"] + "/finetuned_model"
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)

print(f"Fine-tuned model saved to: {output_path}")
print("\nModel includes:")
print("  - LoRA adapter weights")
print("  - Tokenizer configuration")
print("  - Training configuration")

# Finish W&B run
wandb.finish()

print("\nFine-tuning completed successfully!")